# Landlab: grow and erode a mountain range in your browser

[Landlab](https://landlab.csdms.io) is a Python toolkit from [CSDMS](https://csdms.colorado.edu) for 2D models of Earth-surface processes. Here we combine three of its components:

- **uplift**: tectonics lifts the land a little every time step,
- **`FlowAccumulator` + `FastscapeEroder`**: rain water is routed downhill and rivers cut into the rock (stream-power law),
- **`LinearDiffuser`**: soil creeps downhill and smooths the hillslopes.

Starting from a nearly flat, slightly noisy surface, a branching river network and a mountain range emerge.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from landlab import RasterModelGrid, imshow_grid
from landlab.components import FlowAccumulator, FastscapeEroder, LinearDiffuser

## Set up the grid

A 50 x 80 grid of 100 m cells. Water can only leave the domain through the bottom edge.

In [ ]:
grid = RasterModelGrid((50, 80), xy_spacing=100.0)
z = grid.add_zeros("topographic__elevation", at="node")
rng = np.random.default_rng(42)
z += rng.random(z.size)  # a little noise so rivers can find their way

# close right, top and left edges; keep the bottom edge open
grid.set_closed_boundaries_at_grid_edges(True, True, True, False)

flow = FlowAccumulator(grid, flow_director="D8", depression_finder="DepressionFinderAndRouter")
rivers = FastscapeEroder(grid, K_sp=1e-5, m_sp=0.5, n_sp=1.0)
hillslopes = LinearDiffuser(grid, linear_diffusivity=0.01)

uplift_rate = 0.001  # m / yr
dt = 1000.0  # yr

## Run the model

200 steps of 1000 years each, keeping a few snapshots along the way.

In [ ]:
snapshots = {}
for step in range(1, 201):
    z[grid.core_nodes] += uplift_rate * dt
    flow.run_one_step()
    rivers.run_one_step(dt)
    hillslopes.run_one_step(dt)
    if step in (20, 60, 200):
        snapshots[step] = z.copy()

print(f"max elevation after {step * dt / 1000:.0f} kyr: {z.max():.0f} m")

In [ ]:
fig = plt.figure(figsize=(15, 4))
for i, (step, elevation) in enumerate(snapshots.items()):
    plt.subplot(1, 3, i + 1)
    imshow_grid(grid, elevation, cmap="terrain", colorbar_label="elevation (m)")
    plt.title(f"after {step * dt / 1000:.0f} kyr")
plt.tight_layout()

## The river network

The drainage area (how much land drains through each cell) highlights the rivers.

In [ ]:
plt.figure(figsize=(8, 5))
imshow_grid(
    grid,
    np.log10(grid.at_node["drainage_area"] + 1.0),
    cmap="Blues",
    colorbar_label="log10 drainage area (m²)",
)
plt.title("river network")

## A view from the side

A 3D surface of the final landscape.

In [ ]:
x = grid.x_of_node.reshape(grid.shape) / 1000.0
y = grid.y_of_node.reshape(grid.shape) / 1000.0
elev = z.reshape(grid.shape)

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(projection="3d")
ax.plot_surface(x, y, elev, cmap="terrain", linewidth=0, antialiased=True)
ax.set_xlabel("x (km)")
ax.set_ylabel("y (km)")
ax.set_zlabel("elevation (m)")
ax.view_init(elev=35, azim=-60)

Try changing `uplift_rate`, `K_sp` (river erodibility) or `linear_diffusivity` and rerun the model. Many more examples are in the [landlab tutorials](https://landlab.csdms.io/tutorials/index.html).